# Module 04 — Notebook 1: NumPy Arrays

## Learning Objectives

By the end of this notebook you will be able to:
- Create NumPy arrays from Python lists
- Understand why arrays are faster than loops for math
- Perform vectorized arithmetic without explicit loops
- Compute summary statistics: mean, std, min, max, sum, median
- Use boolean indexing to filter arrays
- Classify values with `np.where` and find positions with `np.argmax` / `np.argmin`

**Time:** ~20 minutes

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_length
import numpy as np

print("NumPy version:", np.__version__)

## 1. What is a NumPy Array?

In JavaScript you might write:
```javascript
const scores = [0.92, 0.96, 0.88, 0.94, 0.78];
const doubled = scores.map(x => x * 2);
```

That `.map()` calls a JavaScript function for every element. For 10 million scores, that's 10 million function calls.

NumPy arrays skip the function call overhead — multiplication happens in compiled C code across the whole array at once. For 10 million scores, that's the difference between seconds and milliseconds.

| Operation | Python list | NumPy array |
|-----------|-------------|-------------|
| Type | `list` | `numpy.ndarray` |
| JS analogue | `Array` | `Float64Array` (but smarter) |
| Multiply all by 2 | `[x*2 for x in lst]` | `arr * 2` |
| Sum all | `sum(lst)` | `np.sum(arr)` |
| Filter above 0.8 | `[x for x in lst if x > 0.8]` | `arr[arr > 0.8]` |

A NumPy array is **typed** (all elements have the same dtype, usually `float64`) and **fixed-size** — unlike a Python list which can hold anything.

In [ ]:
import numpy as np

# From a Python list — the most common way
scores = np.array([0.92, 0.96, 0.88, 0.94, 0.78])
print(scores)
print("type: ", type(scores))       # <class 'numpy.ndarray'>
print("dtype:", scores.dtype)       # float64
print("shape:", scores.shape)       # (5,)  — a 1D array with 5 elements
print("len:  ", len(scores))        # 5

# Utility constructors
zeros = np.zeros(5)                 # [0. 0. 0. 0. 0.]
ones  = np.ones(5)                  # [1. 1. 1. 1. 1.]
print("\nzeros:", zeros)
print("ones: ", ones)

# Integer range — like Array.from({length: N}, (_, i) => i)
indices = np.arange(5)              # [0 1 2 3 4]
print("arange:", indices)

# Evenly spaced floats
thresholds = np.linspace(0.0, 1.0, 5)  # [0.   0.25 0.5  0.75 1.  ]
print("linspace:", thresholds)

## 2. Vectorized Operations

**The key idea:** arithmetic on a NumPy array applies to every element — no loop needed.

```javascript
// JavaScript: need .map()
scores.map(x => x * 100)   // → array of percentages
scores.map(x => 1 - x)     // → error rates
```

```python
# Python/NumPy: just use the operator
scores * 100     # → array of percentages
1 - scores       # → error rates
```

You can also do element-wise operations between two same-size arrays — like a `zip().map()` in JS.

In [ ]:
scores = np.array([0.92, 0.96, 0.88, 0.94, 0.78, 0.71, 0.60, 0.65, 0.82, 0.74,
                   0.95, 0.98, 0.91, 0.96, 0.83, 0.78, 0.72, 0.70, 0.85, 0.77])

# Arithmetic on the whole array at once
scores_pct = scores * 100             # convert to percentages
error_rates = 1 - scores              # complement
adjusted    = scores + 0.05           # add a bonus

print("Original (first 5): ", scores[:5])
print("As percentages:     ", scores_pct[:5])
print("Error rates:        ", error_rates[:5])
print("After +0.05:        ", adjusted[:5])

# Element-wise between two arrays
model_a_v2 = np.array([0.95, 0.98, 0.91, 0.96, 0.83])
model_b_v1 = np.array([0.71, 0.60, 0.65, 0.82, 0.74])
improvement = model_a_v2 - model_b_v1
print("\nImprovement (a-v2 minus b-v1):", improvement)
print("Mean improvement:             ", round(float(np.mean(improvement)), 3))

## 3. Summary Statistics

NumPy has built-in functions for all common stats. In JavaScript you'd need to write these yourself or use a library.

| Function | What it computes |
|----------|------------------|
| `np.mean(arr)` | arithmetic mean |
| `np.std(arr)` | standard deviation (population, ddof=0) |
| `np.min(arr)` | minimum value |
| `np.max(arr)` | maximum value |
| `np.sum(arr)` | sum of all elements |
| `np.median(arr)` | median value |
| `np.percentile(arr, q)` | q-th percentile |

> **Heads-up:** `np.std` uses population std (divides by N). pandas `.std()` uses sample std (divides by N-1). For small datasets the difference is noticeable. You'll see both in research code.

In [ ]:
scores = np.array([0.92, 0.96, 0.88, 0.94, 0.78, 0.71, 0.60, 0.65, 0.82, 0.74,
                   0.95, 0.98, 0.91, 0.96, 0.83, 0.78, 0.72, 0.70, 0.85, 0.77])

print(f"Count:   {len(scores)}")
print(f"Mean:    {np.mean(scores):.4f}")
print(f"Std:     {np.std(scores):.4f}")
print(f"Min:     {np.min(scores)}")
print(f"Max:     {np.max(scores)}")
print(f"Median:  {np.median(scores)}")
print(f"25th %:  {np.percentile(scores, 25)}")
print(f"75th %:  {np.percentile(scores, 75)}")

# np.sum on a boolean array counts the Trues
print(f"\nScores above 0.9: {int(np.sum(scores > 0.9))}")

## 4. Boolean Indexing

This is one of the most useful NumPy patterns. In JavaScript:

```javascript
scores.filter(s => s > 0.9)       // → new array of matching elements
```

In NumPy, you do it in two steps (or one):

```python
mask = scores > 0.9               # Step 1: boolean array [False, True, False, ...]
high = scores[mask]               # Step 2: keep only Trues

# Or in one line:
high = scores[scores > 0.9]
```

**Combining conditions:** use `&` (and), `|` (or), with parentheses around each condition.
Python's `and` / `or` keywords do **not** work here — they're for scalars, not arrays.

In [ ]:
scores = np.array([0.92, 0.96, 0.88, 0.94, 0.78, 0.71, 0.60, 0.65, 0.82, 0.74,
                   0.95, 0.98, 0.91, 0.96, 0.83, 0.78, 0.72, 0.70, 0.85, 0.77])

# Step 1: create a boolean mask
mask = scores > 0.9
print("Mask:", mask)

# Step 2: use the mask to select elements
high = scores[mask]
print("High scores:", high)
print("Count:", len(high))

# Combining conditions — use & not 'and', with parentheses
mid = scores[(scores >= 0.7) & (scores <= 0.85)]
print("\nMid-range (0.7–0.85):", mid)

# np.where: classify every element — like a ternary operator across the array
labels = np.where(scores > 0.85, "high", "low")
print("\nLabels:", labels)

# np.argmax / np.argmin: index of the largest / smallest element
print("\nIndex of max:", np.argmax(scores), "→ score:", scores[np.argmax(scores)])
print("Index of min:", np.argmin(scores), "→ score:", scores[np.argmin(scores)])

---
## Your Turn — Exercise 1: Create an Array and Compute Stats

The scores below are from **model-a-v2** across its five evaluation tasks.

1. Create a NumPy array called `a2_scores` from the list `raw`.
2. Compute the mean and store it in `a2_mean`, rounded to **3 decimal places**.
3. Compute the max and store it in `a2_max`.

In [ ]:
raw = [0.95, 0.98, 0.91, 0.96, 0.83]

# YOUR CODE HERE
a2_scores = None   # create np.array from raw
a2_mean   = None   # mean of a2_scores, rounded to 3 decimal places
a2_max    = None   # max of a2_scores

In [ ]:
check_type(a2_scores, np.ndarray, "a2_scores is an ndarray")
check_length(a2_scores, 5, "a2_scores has 5 elements")
check_approx(a2_mean, 0.926, tolerance=1e-3, label="a2_mean")
check_approx(a2_max, 0.98, tolerance=1e-6, label="a2_max")

---
## Your Turn — Exercise 2: Boolean Indexing

Using the full `scores` array (all 20 evaluation scores, provided below):

1. Create a boolean mask `passing_mask` for scores **strictly above 0.8**.
2. Use it to create `passing_scores`, the filtered array.
3. Store the count of passing scores in `n_passing`.

In [ ]:
scores = np.array([0.92, 0.96, 0.88, 0.94, 0.78, 0.71, 0.60, 0.65, 0.82, 0.74,
                   0.95, 0.98, 0.91, 0.96, 0.83, 0.78, 0.72, 0.70, 0.85, 0.77])

# YOUR CODE HERE
passing_mask   = None   # boolean mask: scores > 0.8
passing_scores = None   # array of scores that pass
n_passing      = None   # integer count

In [ ]:
check_type(passing_mask, np.ndarray, "passing_mask is an ndarray")
check_type(passing_scores, np.ndarray, "passing_scores is an ndarray")
check_equal(int(n_passing), 11, "11 scores above 0.8")
check_length(passing_scores, 11, "passing_scores has 11 elements")

---
## Your Turn — Exercise 3: Vectorized Improvement

model-a-v2 and model-b-v1 have scores on the same 5 tasks (in the same order).

1. Create `improvement` as the element-wise difference (`a2` minus `b1`).
2. Store the mean improvement in `mean_improvement`, rounded to **3 decimal places**.
3. Store the **0-based index** of the task with the largest improvement in `best_task_idx`.

> **Hint:** `np.argmax()` returns the index of the maximum element.

In [ ]:
a2 = np.array([0.95, 0.98, 0.91, 0.96, 0.83])  # model-a-v2
b1 = np.array([0.71, 0.60, 0.65, 0.82, 0.74])  # model-b-v1

# YOUR CODE HERE
improvement      = None   # element-wise difference
mean_improvement = None   # mean of improvement, rounded to 3 decimal places
best_task_idx    = None   # index where improvement is largest

In [ ]:
check_type(improvement, np.ndarray, "improvement is an ndarray")
check_approx(mean_improvement, 0.222, tolerance=1e-3, label="mean_improvement")
check_equal(int(best_task_idx), 1, "largest improvement at index 1 (harmful_refusal)")

---
## Why This Matters for AI Research Engineering

Evaluation pipelines regularly process thousands of model outputs. Pure Python loops are too slow for this at scale — NumPy is what makes it fast.

Concretely:
- `np.mean(scores)` computes the average benchmark score across an entire test suite in one line
- `scores[scores < 0.7]` immediately surfaces failing conditions without a loop
- `np.where(scores > threshold, "pass", "fail")` turns a score array into a label array — the kind of operation you do on every row of an evaluation result table
- `np.argmin(scores)` finds the worst-performing case to investigate first

You'll also work with NumPy arrays directly when computing **cosine similarity** between embedding vectors, **z-scores** for cross-task normalization, and **bootstrap confidence intervals** over evaluation metrics. All of those are just vectorized NumPy math.

## Summary

| What | Code |
|------|------|
| Create array | `np.array([1, 2, 3])` |
| Zeros / ones / range | `np.zeros(n)`, `np.ones(n)`, `np.arange(n)` |
| Arithmetic (no loop) | `arr * 2`, `arr + 0.05`, `arr1 - arr2` |
| Stats | `np.mean(arr)`, `np.std(arr)`, `np.min(arr)`, `np.max(arr)` |
| Median / percentile | `np.median(arr)`, `np.percentile(arr, 25)` |
| Boolean mask | `arr > 0.9` → array of True/False |
| Filter | `arr[arr > 0.9]` |
| Count Trues | `np.sum(arr > 0.9)` |
| Classify | `np.where(arr > 0.9, "high", "low")` |
| Index of max/min | `np.argmax(arr)`, `np.argmin(arr)` |

**Next:** Notebook 2 — pandas DataFrames, where you'll load evaluation_results.csv and work with labeled tabular data.